# 資料前處理(Missing data, One-hot encoding, Feature Scaling)
資料的品質、特徵的選取決定了機器學習的上限，模型(Model)只是逼近這個上限。<br>
雖然在學術界總是以Model為主要討論對象，但實際上在業界80%的時間都是在對資料進行前處理，<br>
包含了資料獲取、清理、特徵選擇、特徵處理…到這裡我們可以稍微了解資料前處理的重要性。<br>

常見的資料前處理如下所示：<br>
1.缺失值的處理<br>
2.類別資料的處理（有序、無序）<br>
3.資料特徵縮放<br>

In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from io import StringIO
import math

## 1. 缺失值的處理
缺值主要處理方式有兩種：<br>
*丟棄，如果資料量夠多<br>
*補值<br>

In [2]:
csv_data = '''A,B,C,D,E
            5.0,2.0,3.0,,6
            1.0,6.0,,8.0,5
            0.0,11.0,12.0,4.0,5
            3.0,,3.0,5.0,
            5.0,1.0,4.0,2.0,4
           '''
df = pd.read_csv(StringIO(csv_data))

In [3]:
df = pd.read_csv('preText.csv')

In [4]:
df

,A,B,C,D,E
0,5.0,2.0,3.0,NaN,6.0
1,1.0,6.0,NaN,8.0,5.0
2,0.0,11.0,12.0,4.0,5.0
3,3.0,NaN,3.0,5.0,NaN
4,5.0,1.0,4.0,2.0,4.0


## Missing Data(空值資料處理)

使用Pandas去空值的方法“dropna”，dropna預設只要任一欄位有空值，就會整筆刪掉。<br>
可透過參數來調整，像是把how設為all，就是要全部為空才清掉，或是用subset指定當某一欄為空時才刪。

In [5]:
# 各欄有幾個空值
df.isnull().sum()

A    0
B    1
C    1
D    1
E    1
dtype: int64

In [6]:
# 有空值的列直接刪掉
df.dropna()

,A,B,C,D,E
2,0.0,11.0,12.0,4.0,5.0
4,5.0,1.0,4.0,2.0,4.0


In [7]:
# 只有整列都是空值時才刪除
df.dropna(how='all')

,A,B,C,D,E
0,5.0,2.0,3.0,NaN,6.0
1,1.0,6.0,NaN,8.0,5.0
2,0.0,11.0,12.0,4.0,5.0
3,3.0,NaN,3.0,5.0,NaN
4,5.0,1.0,4.0,2.0,4.0


補值則使用fillna函式即可，依照以下範例即可補上固定值0、平均數、眾數、中位數…

In [8]:
# 空值全部補 0
df.fillna(0)

,A,B,C,D,E
0,5.0,2.0,3.0,0.0,6.0
1,1.0,6.0,0.0,8.0,5.0
2,0.0,11.0,12.0,4.0,5.0
3,3.0,0.0,3.0,5.0,0.0
4,5.0,1.0,4.0,2.0,4.0


In [9]:
# 用各欄平均值補空值
df.fillna(df.mean())

,A,B,C,D,E
0,5.0,2.0,3.0,4.75,6.0
1,1.0,6.0,5.5,8.00,5.0
2,0.0,11.0,12.0,4.00,5.0
3,3.0,5.0,3.0,5.00,5.0
4,5.0,1.0,4.0,2.00,4.0


In [10]:
# 用各欄中位數補空值
df.fillna(df.median())

,A,B,C,D,E
0,5.0,2.0,3.0,4.5,6.0
1,1.0,6.0,3.5,8.0,5.0
2,0.0,11.0,12.0,4.0,5.0
3,3.0,4.0,3.0,5.0,5.0
4,5.0,1.0,4.0,2.0,4.0


In [11]:
# 用各欄眾數補空值
df.fillna(df.mode().iloc[0])

,A,B,C,D,E
0,5.0,2.0,3.0,2.0,6.0
1,1.0,6.0,3.0,8.0,5.0
2,0.0,11.0,12.0,4.0,5.0
3,3.0,1.0,3.0,5.0,5.0
4,5.0,1.0,4.0,2.0,4.0


In [12]:
# 比較補值前後剩下多少空值
filled_df = df.fillna(df.mean())
print(f"補值前空值：{df.isnull().sum().sum()}")
print(f"補值後空值：{filled_df.isnull().sum().sum()}")

補值前空值：4
補值後空值：0


## 2.Categorical Data(類別資料處理)
由於要在空間中表示點，所有的特徵都需要是數值，因此如果是類別的資料，<br>
像是XL,L,M,S,XS或是資料類別為Male, Female, Not Specified，就需要轉成數值才能在空間中來表示。<br>
有序的類別資料通常是直接使用數值替換，比方說XL, L, M, S, XS雖然是類別的屬性但因為有大小順序的關係，可以用10, 7 , 5, 3, 1來替換。<br>如果是Male, Female, Not Specified因為這三種都是等價的關係因此需要找一個方法讓這三個屬性距離原點是相同距離，<br>
One-hot encoding 就是解決這的問題的方法，首先會將Male, Female, Not Specified由Gender從成一個欄位拆成三個欄位，<br>
因此編號1的使用者的屬性資料就是(1,0,0)編號2的使用者就是(0,1,0) 編號三個使用者就是(0,0,1)這三個使用者對於原點的距離都是1，就達成我們想要的結果了。<br>
但One-hot encoding的方法只適合類別種類少的形況下，如果類別種類太多就會產生出一大堆的特徵，造成其他的問題（比方說維數災難）。

In [13]:
df2 = pd.DataFrame(
    [['green', 'M', 10.1, 1],
    ['red', 'L', 13.5, 2],
    ['blue', 'XL', 15.3, 1]]
)
df2.columns = ['color', 'size', 'price', 'classlabel']
df2

,color,size,price,classlabel
0,green,M,10.1,1
1,red,L,13.5,2
2,blue,XL,15.3,1


由於size是屬於有序的資料，我們只要稍微轉換為數值即可

In [14]:
size_mapping = {'XL':10, 'L':7, 'M':5, 'S':3, 'XS':1}
df2['size'] = df2['size'].map(size_mapping)
df2

,color,size,price,classlabel
0,green,5,10.1,1
1,red,7,13.5,2
2,blue,10,15.3,1


至於color我們就用onehot-encoding的方法來處理，在pandas裡面要使用<br>
onehot-encoding使用get_dummies這個函式就可以了，範例如下


In [15]:
# color 是沒有大小順序的類別
pd.get_dummies(df2, columns=['color'], dtype=int)

,size,price,classlabel,color_blue,color_green,color_red
0,5,10.1,1,0,1,0
1,7,13.5,2,0,0,1
2,10,15.3,1,1,0,0


In [16]:
# 也可以只轉 color 這一欄
pd.get_dummies(df2['color'], dtype=int)

,blue,green,red
0,0,1,0
1,0,0,1
2,1,0,0


In [17]:
# drop_first 可少產生一個類別欄位
pd.get_dummies(df2, columns=['color'], drop_first=True, dtype=int)

,size,price,classlabel,color_green,color_red
0,5,10.1,1,1,0
1,7,13.5,2,0,1
2,10,15.3,1,0,0


In [18]:
# 轉好後仍保留原來的 price、classlabel 資料
encoded_df = pd.get_dummies(df2, columns=['color'], dtype=int)
encoded_df

,size,price,classlabel,color_blue,color_green,color_red
0,5,10.1,1,0,1,0
1,7,13.5,2,0,0,1
2,10,15.3,1,1,0,0


# 3.資料特徵縮放
## 資料正規化(normalization)
最常見的Normalization為0–1區間縮放，經過Normalization之後資料的範圍會介在0~1之間，<br>
原本的最大值變為1，最小值變為0，具體作法如下圖


In [19]:
from IPython.display import Math

In [20]:
Math(r'x^{(i)}_{norm}=\frac{x^{(i)}-x_{min}}{x_{max}-x_{min}}')



<IPython.core.display.Math object>

In [21]:
import numpy as np
import pandas as pd
from sklearn import datasets
from io import StringIO

iris = datasets.load_iris()
x = pd.DataFrame(iris['data'], columns=iris['feature_names'])
print("target_names: "+str(iris['target_names']))
y = pd.DataFrame(iris['target'], columns=['target_names'])
data = pd.concat([x,y], axis=1)
data.head(3)

target_names: ['setosa' 'versicolor' 'virginica']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target_names
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0


In [22]:
# 用 NumPy / Pandas 的陣列運算完成 0~1 正規化
x_min = x.min()
x_max = x.max()
x_norm = (x - x_min) / (x_max - x_min)
x_norm.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,0.222222,0.625000,0.067797,0.041667
1,0.166667,0.416667,0.067797,0.041667
2,0.111111,0.500000,0.050847,0.041667
3,0.083333,0.458333,0.084746,0.041667
4,0.194444,0.666667,0.067797,0.041667


## 資料標準化(Standardization)
經過Standardization資料的平均值會變為0, 標準差變為1

In [23]:
Math(r'x^{(i)}_{std}=\frac{x^{(i)}-\mu_{x}}{\sigma_{x}}')



<IPython.core.display.Math object>

In [24]:
# 各欄減平均數，再除以標準差
x_std = (x - x.mean()) / x.std(ddof=0)
x_std.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,-0.900681,1.019004,-1.340227,-1.315444
1,-1.143017,-0.131979,-1.340227,-1.315444
2,-1.385353,0.328414,-1.397064,-1.315444
3,-1.506521,0.098217,-1.283389,-1.315444
4,-1.021849,1.249201,-1.340227,-1.315444


In [25]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

minmax_scaler = MinMaxScaler()
standard_scaler = StandardScaler()

print("MinMaxScaler 前 3 筆：")
print(minmax_scaler.fit_transform(x)[:3])
print("StandardScaler 前 3 筆：")
print(standard_scaler.fit_transform(x)[:3])

MinMaxScaler 前 3 筆：
[[0.22222222 0.625      0.06779661 0.04166667]
 [0.16666667 0.41666667 0.06779661 0.04166667]
 [0.11111111 0.5        0.05084746 0.04166667]]
StandardScaler 前 3 筆：
[[-0.90068117  1.01900435 -1.34022653 -1.3154443 ]
 [-1.14301691 -0.13197948 -1.34022653 -1.3154443 ]
 [-1.38535265  0.32841405 -1.39706395 -1.3154443 ]]
